In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('train.txt',sep = ';',header = None,names = ['text','emotion'])

In [3]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [4]:
df.isnull().sum()

,0
text,0
emotion,0


In [5]:
df['text'] = df['text'].apply(lambda x : x.lower())

In [6]:
import string

def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))

In [7]:
df['text'] = df['text'].apply(remove_punc)

In [8]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)


In [9]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)


In [10]:
import nltk

In [11]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize


In [12]:

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [13]:
stop_words = set(stopwords.words('english'))

In [14]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [16]:

def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)

  return ' '.join(cleaned)

In [17]:
df['text'] = df['text'].apply(remove)

In [18]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [19]:
df.head()

,text,emotion
0,didnt feel humiliated,sadness
1,go feeling hopeless damned hopeful around some...,sadness
2,im grabbing minute post feel greedy wrong,anger
3,ever feeling nostalgic fireplace know still pr...,love
4,feeling grouchy,anger


In [20]:
print(df["emotion"].unique())

['sadness' 'anger' 'love' 'surprise' 'fear' 'joy']


In [21]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y = le.fit_transform(df["emotion"])

print("Emotion mapping:")

for i, emotion in enumerate(le.classes_):
    print(f"{i} → {emotion}")

Emotion mapping:
0 → anger
1 → fear
2 → joy
3 → love
4 → sadness
5 → surprise


In [22]:
print("Original labels:")
print(df["emotion"].unique())

print("\nEncoded labels:")
print(y[:10])

Original labels:
['sadness' 'anger' 'love' 'surprise' 'fear' 'joy']

Encoded labels:
[4 4 0 3 0 4 5 1 2 3]


In [23]:
from sklearn.model_selection import train_test_split

X = df["text"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

BOW model

In [24]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# BOW
bow_vectorizer = CountVectorizer()

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

# Model
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

# Prediction
pred_bow = nb_model.predict(X_test_bow)

# Evaluation
accuracy = accuracy_score(y_test, pred_bow)

print("BOW + Naive Bayes Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(y_test, pred_bow))

BOW + Naive Bayes Accuracy: 0.7803125

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.66      0.76       432
           1       0.86      0.63      0.73       387
           2       0.75      0.93      0.83      1072
           3       0.88      0.31      0.46       261
           4       0.76      0.94      0.84       933
           5       0.79      0.10      0.17       115

    accuracy                           0.78      3200
   macro avg       0.82      0.59      0.63      3200
weighted avg       0.80      0.78      0.76      3200



TF-IDF + Naive Bayes

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# TF-IDF
tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Model
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)

# Prediction
pred_tfidf = nb_tfidf.predict(X_test_tfidf)

# Evaluation
accuracy_tfidf = accuracy_score(y_test, pred_tfidf)

print("TF-IDF + Naive Bayes Accuracy:", accuracy_tfidf)
print("\nClassification Report:")
print(classification_report(y_test, pred_tfidf))

TF-IDF + Naive Bayes Accuracy: 0.670625

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.29      0.44       432
           1       0.94      0.26      0.41       387
           2       0.62      0.98      0.76      1072
           3       1.00      0.06      0.12       261
           4       0.68      0.92      0.78       933
           5       0.00      0.00      0.00       115

    accuracy                           0.67      3200
   macro avg       0.70      0.42      0.42      3200
weighted avg       0.73      0.67      0.60      3200



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


BOW + Logistic Regression

In [26]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# BOW
bow_vectorizer = CountVectorizer()

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

# Logistic Regression
lr_model = LogisticRegression(
    max_iter=1000
)

lr_model.fit(X_train_bow, y_train)

# Prediction
pred_lr_bow = lr_model.predict(X_test_bow)

# Evaluation
print("BOW + Logistic Regression Accuracy:",
      accuracy_score(y_test, pred_lr_bow))

print("\nClassification Report:")
print(classification_report(y_test, pred_lr_bow))

BOW + Logistic Regression Accuracy: 0.8896875

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.85      0.87       432
           1       0.87      0.86      0.86       387
           2       0.89      0.93      0.91      1072
           3       0.79      0.77      0.78       261
           4       0.93      0.93      0.93       933
           5       0.83      0.75      0.79       115

    accuracy                           0.89      3200
   macro avg       0.87      0.85      0.86      3200
weighted avg       0.89      0.89      0.89      3200



TF-IDF + Logistic Regression

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

lr_tfidf = LogisticRegression(
    max_iter=1000
)

lr_tfidf.fit(X_train_tfidf, y_train)

pred_lr_tfidf = lr_tfidf.predict(X_test_tfidf)

print(
    "TF-IDF + Logistic Regression Accuracy:",
    accuracy_score(y_test, pred_lr_tfidf)
)

print("\nClassification Report:")
print(classification_report(y_test, pred_lr_tfidf))

TF-IDF + Logistic Regression Accuracy: 0.86375

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.78      0.84       432
           1       0.88      0.80      0.83       387
           2       0.82      0.95      0.88      1072
           3       0.87      0.63      0.73       261
           4       0.90      0.94      0.92       933
           5       0.93      0.55      0.69       115

    accuracy                           0.86      3200
   macro avg       0.88      0.77      0.81      3200
weighted avg       0.87      0.86      0.86      3200



BOW + Linear SVM

In [28]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

# BOW
bow_vectorizer_svm = CountVectorizer()

X_train_bow_svm = bow_vectorizer_svm.fit_transform(X_train)
X_test_bow_svm = bow_vectorizer_svm.transform(X_test)

# Linear SVM
svm_model = LinearSVC()

svm_model.fit(X_train_bow_svm, y_train)

# Prediction
pred_svm_bow = svm_model.predict(X_test_bow_svm)

# Evaluation
print(
    "BOW + Linear SVM Accuracy:",
    accuracy_score(y_test, pred_svm_bow)
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        pred_svm_bow
    )
)

BOW + Linear SVM Accuracy: 0.888125

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.86      0.87       432
           1       0.87      0.87      0.87       387
           2       0.91      0.91      0.91      1072
           3       0.76      0.81      0.78       261
           4       0.92      0.92      0.92       933
           5       0.83      0.77      0.80       115

    accuracy                           0.89      3200
   macro avg       0.86      0.86      0.86      3200
weighted avg       0.89      0.89      0.89      3200



TF-IDF + Linear SVM

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

# TF-IDF
tfidf_vectorizer_svm = TfidfVectorizer()

X_train_tfidf_svm = tfidf_vectorizer_svm.fit_transform(X_train)
X_test_tfidf_svm = tfidf_vectorizer_svm.transform(X_test)

# Linear SVM
svm_tfidf = LinearSVC()

svm_tfidf.fit(X_train_tfidf_svm, y_train)

# Prediction
pred_svm_tfidf = svm_tfidf.predict(X_test_tfidf)

# Evaluation
print(
    "TF-IDF + Linear SVM Accuracy:",
    accuracy_score(y_test, pred_svm_tfidf)
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        pred_svm_tfidf
    )
)

TF-IDF + Linear SVM Accuracy: 0.89

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.87      0.87       432
           1       0.87      0.87      0.87       387
           2       0.89      0.93      0.91      1072
           3       0.79      0.77      0.78       261
           4       0.93      0.92      0.93       933
           5       0.86      0.72      0.79       115

    accuracy                           0.89      3200
   macro avg       0.87      0.85      0.86      3200
weighted avg       0.89      0.89      0.89      3200



In [30]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

final_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", LinearSVC())
])

In [31]:
final_pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('classifier', LinearSVC())])

In [32]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = final_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.89

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.87      0.87       432
           1       0.87      0.87      0.87       387
           2       0.89      0.93      0.91      1072
           3       0.79      0.77      0.78       261
           4       0.93      0.92      0.93       933
           5       0.86      0.72      0.79       115

    accuracy                           0.89      3200
   macro avg       0.87      0.85      0.86      3200
weighted avg       0.89      0.89      0.89      3200



In [33]:
import os
import joblib

os.makedirs("models", exist_ok=True)

joblib.dump(
    final_pipeline,
    "models/emotion_classifier.pkl"
)

joblib.dump(
    le,
    "models/label_encoder.pkl"
)

print("Model and label encoder saved successfully.")

Model and label encoder saved successfully.


In [34]:
import joblib

loaded_model = joblib.load(
    "models/emotion_classifier.pkl"
)

loaded_encoder = joblib.load(
    "models/label_encoder.pkl"
)

text = "I am extremely happy today!"

prediction = loaded_model.predict([text])[0]

emotion = loaded_encoder.inverse_transform(
    [prediction]
)[0]

print("Predicted class:", prediction)
print("Predicted emotion:", emotion)

Predicted class: 2
Predicted emotion: joy


In [35]:
print(df.head())
print(df.columns)

                                                text  emotion
0                              didnt feel humiliated  sadness
1  go feeling hopeless damned hopeful around some...  sadness
2          im grabbing minute post feel greedy wrong    anger
3  ever feeling nostalgic fireplace know still pr...     love
4                                    feeling grouchy    anger
Index(['text', 'emotion'], dtype='object')


In [36]:
print(df["emotion"].unique())

['sadness' 'anger' 'love' 'surprise' 'fear' 'joy']


In [37]:
print(df["emotion"].value_counts())

emotion
joy         5362
sadness     4666
anger       2159
fear        1937
love        1304
surprise     572
Name: count, dtype: int64


In [38]:
print(sorted(df["emotion"].unique()))

['anger', 'fear', 'joy', 'love', 'sadness', 'surprise']


In [39]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y = le.fit_transform(df["emotion"])

print("Emotion mapping:")

for i, emotion in enumerate(le.classes_):
    print(f"{i} → {emotion}")

Emotion mapping:
0 → anger
1 → fear
2 → joy
3 → love
4 → sadness
5 → surprise


In [40]:
print(df.head(20))

                                                 text   emotion
0                               didnt feel humiliated   sadness
1   go feeling hopeless damned hopeful around some...   sadness
2           im grabbing minute post feel greedy wrong     anger
3   ever feeling nostalgic fireplace know still pr...      love
4                                     feeling grouchy     anger
5       ive feeling little burdened lately wasnt sure   sadness
6   ive taking milligrams times recommended amount...  surprise
7      feel confused life teenager jaded year old man      fear
8   petronas years feel petronas performed well ma...       joy
9                                       feel romantic      love
10     feel like make suffering seeing mean something   sadness
11  feel running divine experience expect type spi...       joy
12          think easiest time year feel dissatisfied     anger
13                            feel low energy thirsty   sadness
14  immense sympathy general point possi

In [41]:
print(le.classes_)

['anger' 'fear' 'joy' 'love' 'sadness' 'surprise']


In [42]:
for i, emotion in enumerate(le.classes_):
    print(i, "→", emotion)

0 → anger
1 → fear
2 → joy
3 → love
4 → sadness
5 → surprise


In [43]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("svm", LinearSVC())
])

model.fit(X_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('svm', LinearSVC())])

In [47]:
predictions = model.predict(X_test)

In [48]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))


Accuracy: 0.89
              precision    recall  f1-score   support

           0       0.88      0.87      0.87       432
           1       0.87      0.87      0.87       387
           2       0.89      0.93      0.91      1072
           3       0.79      0.77      0.78       261
           4       0.93      0.92      0.93       933
           5       0.86      0.72      0.79       115

    accuracy                           0.89      3200
   macro avg       0.87      0.85      0.86      3200
weighted avg       0.89      0.89      0.89      3200

